# WavLM 4-Way Comparison

| | Pretrained | Fine-tuned |
|---|---|---|
| **Whole audio** (768-dim) | ✓ | ✓ |
| **Segmented** (1536-dim) | ✓ | ✓ |

All 4 variants trained and evaluated on the same split.

In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================
from pathlib import Path

TRAIN_FOLDERS = ["audios2", "audios4"]
TEST_FOLDER   = "audios5"   # leave "" for 20% random holdout

PRETRAINED_PATH = "microsoft/wavlm-base-plus"
FINETUNED_PATH  = "wavlm_finetuned"   # local folder next to this notebook

WINDOW_SEC  = 10
HOP_SEC     = 5
MIN_WIN_SEC = 4
TEST_RATIO  = 0.20
RANDOM_SEED = 42
SR          = 16000
AUDIO_EXTS  = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

NB_DIR   = Path(".").resolve()
SAVE_DIR = NB_DIR / "checkpoints_4way"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import warnings, json as _json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import xgboost as xgb
import joblib
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              accuracy_score, confusion_matrix)
from transformers import AutoFeatureExtractor, WavLMModel

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

LABEL_MAP = {
    "cheating": 1, "read": 1, "scripted": 1, "yes": 1, "1": 1,
    "not cheating": 0, "spontaneous": 0, "no": 0, "0": 0,
}

## 1. Scan Folders

In [ ]:
def scan_folder(name):
    audio_dir = NB_DIR / name
    if not audio_dir.exists(): return None
    files = sorted(f for f in audio_dir.rglob('*') if f.suffix.lower() in AUDIO_EXTS)
    if not files: return None
    gt_path = NB_DIR / f'{name}GT.csv'
    if not gt_path.exists(): return None
    return {
        "name":   name,
        "files":  files,
        "gt":     gt_path,
        "csvs": {
            "whole_pre":  NB_DIR / f"{name}_whole_pretrained.csv",
            "whole_ft":   NB_DIR / f"{name}_whole_finetuned.csv",
            "seg_pre":    NB_DIR / f"{name}_seg_pretrained.csv",
            "seg_ft":     NB_DIR / f"{name}_seg_finetuned.csv",
        }
    }

all_names = list(dict.fromkeys(TRAIN_FOLDERS + ([TEST_FOLDER] if TEST_FOLDER else [])))
folders   = [m for m in (scan_folder(n) for n in all_names) if m]

for m in folders:
    status = {k: 'ok' if v.exists() else 'needed' for k, v in m['csvs'].items()}
    print(f"{m['name']:12s}  {len(m['files']):4d} files  {status}")

## 2. Load Both WavLM Models

In [ ]:
def load_model(path):
    print(f'  Loading: {path}')
    fe    = AutoFeatureExtractor.from_pretrained(path)
    model = WavLMModel.from_pretrained(path).eval().to(DEVICE)
    return fe, model

print('Loading pretrained...')
fe_pre, model_pre = load_model(PRETRAINED_PATH)

print('Loading fine-tuned...')
fe_ft,  model_ft  = load_model(FINETUNED_PATH)
print('Both loaded.')

## 3. Extract All 4 Feature Sets

Each folder produces 4 CSVs. Skips any that already exist.

In [ ]:
def load_audio(path):
    try:
        y, sr = sf.read(str(path), always_2d=False)
        if y.ndim > 1: y = y.mean(axis=1)
        if sr != SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=SR)
        return y.astype(np.float32)
    except Exception as e:
        print(f'  WARN {Path(path).name}: {e}'); return None

@torch.no_grad()
def embed_whole(y, fe, model):
    inp = fe(y, sampling_rate=SR, return_tensors='pt', padding=False)
    return model(inp.input_values.to(DEVICE)).last_hidden_state.mean(1).squeeze(0).cpu().numpy()

@torch.no_grad()
def embed_seg(y, fe, model):
    ws = int(WINDOW_SEC * SR); hs = int(HOP_SEC * SR); ms = int(MIN_WIN_SEC * SR)
    wins, start = [], 0
    while start + ws <= len(y): wins.append(y[start:start+ws]); start += hs
    tail = y[start:]
    if len(tail) >= ms: wins.append(tail)
    if not wins: wins = [y]
    embs = []
    for w in wins:
        inp = fe(w, sampling_rate=SR, return_tensors='pt', padding=False)
        embs.append(model(inp.input_values.to(DEVICE)).last_hidden_state.mean(1).squeeze(0).cpu().numpy())
    A = np.array(embs)
    return np.concatenate([A.mean(0), A.std(0)])

def extract_folder(meta):
    needed = {k: v for k, v in meta['csvs'].items() if not v.exists()}
    if not needed:
        print(f"  {meta['name']}: all CSVs exist, skipping."); return

    print(f"\nExtracting {meta['name']} ({len(meta['files'])} files) ...")
    rows = {k: [] for k in needed}

    for fp in tqdm(meta['files'], desc=meta['name']):
        y = load_audio(fp)
        if y is None: continue
        fname = fp.name

        if 'whole_pre' in needed:
            e = embed_whole(y, fe_pre, model_pre)
            row = {'filename': fname}
            for i, v in enumerate(e): row[f'wavlm_{i}'] = round(float(v), 6)
            rows['whole_pre'].append(row)

        if 'whole_ft' in needed:
            e = embed_whole(y, fe_ft, model_ft)
            row = {'filename': fname}
            for i, v in enumerate(e): row[f'wavlm_{i}'] = round(float(v), 6)
            rows['whole_ft'].append(row)

        if 'seg_pre' in needed:
            e = embed_seg(y, fe_pre, model_pre)
            row = {'filename': fname}
            for i, v in enumerate(e[:768]):  row[f'wavlm_mean_{i}'] = round(float(v), 6)
            for i, v in enumerate(e[768:]):  row[f'wavlm_std_{i}']  = round(float(v), 6)
            rows['seg_pre'].append(row)

        if 'seg_ft' in needed:
            e = embed_seg(y, fe_ft, model_ft)
            row = {'filename': fname}
            for i, v in enumerate(e[:768]):  row[f'wavlm_mean_{i}'] = round(float(v), 6)
            for i, v in enumerate(e[768:]):  row[f'wavlm_std_{i}']  = round(float(v), 6)
            rows['seg_ft'].append(row)

    for k, row_list in rows.items():
        if row_list:
            pd.DataFrame(row_list).to_csv(meta['csvs'][k], index=False)
            print(f"  Saved {len(row_list)} rows -> {meta['csvs'][k].name}")

for meta in folders:
    extract_folder(meta)

## 4. Load GT + Build DataFrames

In [ ]:
def load_gt(gt_path):
    gt = pd.read_csv(gt_path)
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt'))
    gt = gt.rename(columns={fn_col: 'filename', lbl_col: 'label_raw'})
    gt['label_int'] = gt['label_raw'].astype(str).str.lower().str.strip().map(LABEL_MAP)
    return gt.dropna(subset=['label_int']).assign(label_int=lambda d: d['label_int'].astype(int))

def build_df(folder_names, csv_key):
    dfs = []
    for name in folder_names:
        meta = next((m for m in folders if m['name'] == name), None)
        if meta is None: continue
        feat = pd.read_csv(meta['csvs'][csv_key])
        gt   = load_gt(meta['gt'])
        merged = feat.merge(gt[['filename','label_int']], on='filename', how='inner')
        merged['batch'] = name
        dfs.append(merged)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

train_names = TRAIN_FOLDERS
test_names  = [TEST_FOLDER] if TEST_FOLDER else []

dfs = {}
for key in ['whole_pre', 'whole_ft', 'seg_pre', 'seg_ft']:
    dfs[f'train_{key}'] = build_df(train_names, key)
    dfs[f'test_{key}']  = build_df(test_names,  key) if test_names else pd.DataFrame()

n = len(dfs['train_whole_pre'])
c = int((dfs['train_whole_pre']['label_int']==1).sum())
print(f'Train: {n} samples  (cheating={c}, honest={n-c})')
if not dfs['test_whole_pre'].empty:
    n = len(dfs['test_whole_pre'])
    c = int((dfs['test_whole_pre']['label_int']==1).sum())
    print(f'Test:  {n} samples from {TEST_FOLDER}  (cheating={c}, honest={n-c})')

## 5. Train / Test Split

In [ ]:
splits = {}

for key in ['whole_pre', 'whole_ft', 'seg_pre', 'seg_ft']:
    tr = dfs[f'train_{key}']
    te = dfs[f'test_{key}']
    feat_cols = [c for c in tr.columns if c.startswith('wavlm_')]

    if not te.empty:
        X_tr = tr[feat_cols].fillna(0).values
        X_te = te[feat_cols].fillna(0).values
        y_tr = tr['label_int'].values
        y_te = te['label_int'].values
    else:
        y_all = tr['label_int'].values
        idx   = np.arange(len(y_all))
        tr_idx, te_idx = train_test_split(idx, test_size=TEST_RATIO,
                                          random_state=RANDOM_SEED, stratify=y_all)
        X_tr = tr.iloc[tr_idx][feat_cols].fillna(0).values
        X_te = tr.iloc[te_idx][feat_cols].fillna(0).values
        y_tr = y_all[tr_idx]
        y_te = y_all[te_idx]

    splits[key] = dict(X_tr=X_tr, X_te=X_te, y_tr=y_tr, y_te=y_te,
                       n_feats=len(feat_cols))

y_tr = splits['whole_pre']['y_tr']
y_te = splits['whole_pre']['y_te']
spw  = float((y_tr==0).sum()) / max(float((y_tr==1).sum()), 1.0)
print(f'scale_pos_weight = {spw:.2f}')
print(f'Train: {int((y_tr==1).sum())} cheating / {int((y_tr==0).sum())} honest')
print(f'Test:  {int((y_te==1).sum())} cheating / {int((y_te==0).sum())} honest')

## 6. Train & Evaluate All 4

In [ ]:
def train_xgb(X_tr, X_te, y_tr, y_te, spw, n_feats):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_tr)
    Xte = scaler.transform(X_te)
    colsample = 0.3 if n_feats > 500 else 0.8
    model = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.04,
        subsample=0.8, colsample_bytree=colsample,
        min_child_weight=3, scale_pos_weight=spw,
        eval_metric='logloss', early_stopping_rounds=30,
        random_state=RANDOM_SEED, device='cpu',
    )
    model.fit(Xtr, y_tr, eval_set=[(Xte, y_te)], verbose=False)
    proba = model.predict_proba(Xte)[:, 1]
    best_thr, best_f1 = 0.5, 0.0
    for thr in np.arange(0.20, 0.81, 0.02):
        f = f1_score(y_te, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    preds = (proba >= best_thr).astype(int)
    cm = confusion_matrix(y_te, preds, labels=[0,1])
    return dict(
        f1=round(best_f1,4),
        precision=round(precision_score(y_te, preds, zero_division=0), 4),
        recall=round(recall_score(y_te, preds, zero_division=0), 4),
        accuracy=round(accuracy_score(y_te, preds), 4),
        threshold=round(best_thr, 2),
        tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0]),
        model=model, scaler=scaler, proba=proba,
    )

results = {}
LABELS = {
    'whole_pre': 'Whole  + Pretrained  (768-dim)',
    'whole_ft':  'Whole  + Fine-tuned  (768-dim)',
    'seg_pre':   'Seg    + Pretrained  (1536-dim)',
    'seg_ft':    'Seg    + Fine-tuned  (1536-dim)',
}

for key, label in LABELS.items():
    print(f'Training {label} ...')
    s = splits[key]
    results[key] = train_xgb(s['X_tr'], s['X_te'], s['y_tr'], s['y_te'],
                              spw, s['n_feats'])
    r = results[key]
    print(f'  F1={r["f1"]:.4f}  Prec={r["precision"]:.4f}  '
          f'Rec={r["recall"]:.4f}  Thr={r["threshold"]:.2f}')

print('Done.')

## 7. Comparison Table + Interpretation

In [ ]:
print(f'\nTrain: {" + ".join(TRAIN_FOLDERS)}   Test: {TEST_FOLDER or "20% holdout"}')
print(f'Test set: {len(y_te)} samples  '
      f'(cheating={int((y_te==1).sum())}, honest={int((y_te==0).sum())})')
print()
print(f'{"="*72}')
print(f'  {"Model":<34} {"Prec":>6} {"Rec":>6} {"F1":>6} {"Thr":>5} {"TP":>4} {"FP":>4} {"FN":>4}')
print(f'  {"-"*34} {"-"*6} {"-"*6} {"-"*6} {"-"*5} {"-"*4} {"-"*4} {"-"*4}')
for key, label in LABELS.items():
    r = results[key]
    print(f'  {label:<34} {r["precision"]:>6.4f} {r["recall"]:>6.4f} '
          f'{r["f1"]:>6.4f} {r["threshold"]:>5.2f} '
          f'{r["tp"]:>4} {r["fp"]:>4} {r["fn"]:>4}')
print(f'{"="*72}')

# Fine-tuning deltas
print()
print('Fine-tuning impact:')
for rep in ['whole', 'seg']:
    dp = results[f'{rep}_ft']['precision'] - results[f'{rep}_pre']['precision']
    dr = results[f'{rep}_ft']['recall']    - results[f'{rep}_pre']['recall']
    df = results[f'{rep}_ft']['f1']        - results[f'{rep}_pre']['f1']
    print(f'  {rep.capitalize():5s}:  '
          f'Prec {dp:+.4f}   Rec {dr:+.4f}   F1 {df:+.4f}')

print()
print('Verdict:')

# Best precision model
best_prec_key = max(results, key=lambda k: results[k]['precision'])
best_f1_key   = max(results, key=lambda k: results[k]['f1'])
ft_helped_whole = results['whole_ft']['precision'] > results['whole_pre']['precision']
ft_helped_seg   = results['seg_ft']['precision']   > results['seg_pre']['precision']

print(f'  Best precision : {LABELS[best_prec_key]}  ({results[best_prec_key]["precision"]:.4f})')
print(f'  Best F1        : {LABELS[best_f1_key]}  ({results[best_f1_key]["f1"]:.4f})')
print()
if ft_helped_whole or ft_helped_seg:
    print('  Fine-tuning improved precision. Use the fine-tuned model going forward.')
else:
    print('  Fine-tuning did NOT improve precision — epoch 2 model may be undertrained.')
    print('  Resume training on Kaggle (2-3 more epochs) and re-run this notebook.')

## 8. Save Models

In [ ]:
for key, label in LABELS.items():
    r = results[key]
    r['model'].save_model(str(SAVE_DIR / f'xgb_{key}.json'))
    joblib.dump(r['scaler'], str(SAVE_DIR / f'scaler_{key}.pkl'))

summary = {
    'train': TRAIN_FOLDERS, 'test': TEST_FOLDER,
    'pretrained': PRETRAINED_PATH, 'finetuned': FINETUNED_PATH,
}
for key in LABELS:
    r = results[key]
    summary[key] = {k: v for k, v in r.items() if k not in ('model','scaler','proba')}

with open(SAVE_DIR / 'results.json', 'w') as f:
    _json.dump(summary, f, indent=2)

print(f'Saved to {SAVE_DIR}/')